# 🧒 Prediksi Status Stunting Balita Menggunakan Machine Learning

**Peran:** Data Scientist & Instruktur Machine Learning
**Tujuan:** Membangun model klasifikasi untuk memprediksi status stunting (stunting / tidak stunting) pada balita berdasarkan data demografis, gizi, kesehatan, dan lingkungan keluarga.

**Dataset:** `dataset_stunting_ml_1000.csv` (1000 baris, 15 kolom)

**Konteks masalah:** Stunting adalah kondisi gagal tumbuh pada anak balita akibat kekurangan gizi kronis, terutama pada 1000 Hari Pertama Kehidupan (HPK). Deteksi dini faktor risiko stunting sangat penting bagi tenaga kesehatan masyarakat dan pembuat kebijakan agar intervensi dapat dilakukan secara tepat sasaran.

**Alur notebook:**
1. Import Library
2. Load Dataset
3. Exploratory Data Analysis (EDA)
4. Data Preprocessing
5. Train-Test Split
6. Pemilihan Model
7. Training Model
8. Evaluasi Model
9. Feature Importance
10. Model Testing (Data Baru)
11. Kesimpulan


## 1. Import Library

Kita mengimpor seluruh library yang dibutuhkan untuk manipulasi data, visualisasi, dan pemodelan machine learning:

- **pandas & numpy** → manipulasi data tabular dan operasi numerik.
- **matplotlib & seaborn** → visualisasi data (histogram, boxplot, heatmap, dsb).
- **scikit-learn** → preprocessing (encoding, scaling), split data, algoritma machine learning, dan metrik evaluasi.
- **warnings** → menyembunyikan warning non-kritis agar output notebook tetap rapi.


In [ ]:
# ============================================================
# IMPORT LIBRARY
# ============================================================
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
    roc_auc_score,
    roc_curve,
)
from sklearn.inspection import permutation_importance

# Pengaturan tampilan agar output rapi dan konsisten
warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 100
pd.set_option("display.max_columns", None)

RANDOM_STATE = 42  # digunakan di seluruh notebook agar hasil dapat direproduksi

print("Seluruh library berhasil diimpor.")


: 

## 2. Load Dataset

Pada tahap ini kita memuat dataset, memeriksa ukuran (shape), struktur tipe data (info), ringkasan statistik (describe), serta memahami arti setiap kolom sebelum masuk ke analisis lebih dalam.

### Deskripsi Fitur

| Kolom | Tipe | Deskripsi |
|---|---|---|
| `id` | Identifier | ID unik responden (tidak dipakai sebagai fitur prediksi) |
| `usia_bulan` | Numerik | Usia anak dalam bulan (12–59 bulan / kategori balita) |
| `jenis_kelamin` | Kategorikal | Jenis kelamin anak (L = Laki-laki, P = Perempuan) |
| `berat_lahir_kg` | Numerik | Berat badan lahir anak (kg) |
| `panjang_lahir_cm` | Numerik | Panjang badan lahir anak (cm) |
| `asi_eksklusif` | Kategorikal | Status pemberian ASI eksklusif (Ya/Tidak) |
| `protein_harian` | Numerik | Estimasi asupan protein harian anak (gram/hari) |
| `frekuensi_makan` | Numerik | Frekuensi makan anak per hari (kali) |
| `tinggi_ibu_cm` | Numerik | Tinggi badan ibu (cm) — proksi faktor genetik & gizi ibu |
| `riwayat_diare` | Numerik | Jumlah kejadian diare anak dalam periode tertentu |
| `pendapatan_keluarga` | Numerik | Pendapatan keluarga per bulan (Rupiah) |
| `sanitasi_layak` | Kategorikal | Akses terhadap sanitasi layak di rumah (Ya/Tidak) |
| `imunisasi_lengkap` | Kategorikal | Status kelengkapan imunisasi dasar anak (Ya/Tidak) |
| `risk_score` | Numerik | Skor risiko komposit (0–100) hasil agregasi beberapa faktor risiko |
| **`status_stunting`** | **Target** | **0 = Tidak Stunting, 1 = Stunting** |

> **Catatan:** Target `status_stunting` bersifat biner sehingga ini merupakan **kasus klasifikasi biner (binary classification)**.


In [ ]:
# Memuat dataset
df = pd.read_csv("dataset_stunting_ml_1000.csv")

# Preview 5 baris pertama
df.head()


In [ ]:
# Ukuran dataset: (jumlah baris, jumlah kolom)
print(f"Jumlah baris  : {df.shape[0]}")
print(f"Jumlah kolom  : {df.shape[1]}")


In [ ]:
# Informasi struktur data: tipe data & jumlah non-null tiap kolom
df.info()


In [ ]:
# Ringkasan statistik untuk seluruh kolom (numerik & kategorikal)
df.describe(include="all").T


**Insight awal:**
- Dataset terdiri dari **1000 observasi** dan **15 kolom** (13 fitur + 1 ID + 1 target).
- Tidak terdapat nilai non-null yang hilang pada tampilan awal `info()` (akan diverifikasi lebih formal di tahap EDA).
- Terdapat kombinasi fitur **numerik** (usia, berat/panjang lahir, protein, pendapatan, dll.) dan **kategorikal biner** (jenis kelamin, ASI eksklusif, sanitasi, imunisasi).
- Kolom `risk_score` memiliki rentang 0–100 dan kemungkinan besar merupakan skor komposit yang sudah memperhitungkan beberapa faktor risiko lain — ini akan kita cermati lebih lanjut saat melihat korelasi.


## 3. Exploratory Data Analysis (EDA)

Pada bagian ini kita akan melakukan eksplorasi data secara menyeluruh: memeriksa data hilang, duplikasi, tipe data, distribusi target, statistik deskriptif, visualisasi (histogram, boxplot, heatmap korelasi), dan analisis outlier. Setiap tahap disertai insight.


### 3.1 Missing Value, Duplicate, dan Tipe Data

In [ ]:
# Cek jumlah missing value per kolom
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({"jumlah_missing": missing, "persentase(%)": missing_pct})
missing_df


In [ ]:
# Cek jumlah baris duplikat (berdasarkan seluruh kolom)
n_duplicate = df.duplicated().sum()
print(f"Jumlah baris duplikat: {n_duplicate}")


In [ ]:
# Cek tipe data tiap kolom
df.dtypes.to_frame(name="tipe_data")


**Insight:**
- **Tidak ditemukan missing value** pada seluruh kolom (0%), sehingga tidak diperlukan imputasi.
- **Tidak ditemukan baris duplikat**, sehingga tidak ada risiko data ganda yang bisa membiaskan model.
- Tipe data sudah cukup sesuai: kolom numerik terbaca sebagai `int64`/`float64`, kolom kategorikal terbaca sebagai teks (`object`/`string`). Kolom kategorikal ini nantinya perlu di-*encode* sebelum masuk ke model machine learning.


### 3.2 Distribusi Target (`status_stunting`)

In [ ]:
# Hitung jumlah & proporsi tiap kelas pada target
target_counts = df["status_stunting"].value_counts()
target_pct = df["status_stunting"].value_counts(normalize=True) * 100

print("Jumlah per kelas:")
print(target_counts)
print("\nPersentase per kelas:")
print(target_pct.round(2))

# Visualisasi distribusi target
fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))

sns.countplot(x="status_stunting", data=df, palette=["#4C72B0", "#DD8452"], ax=ax[0])
ax[0].set_title("Distribusi Jumlah Status Stunting")
ax[0].set_xlabel("Status Stunting (0 = Tidak, 1 = Ya)")
ax[0].set_ylabel("Jumlah Anak")
for p in ax[0].patches:
    ax[0].annotate(f"{int(p.get_height())}", (p.get_x() + p.get_width() / 2, p.get_height()),
                   ha="center", va="bottom")

ax[1].pie(target_counts, labels=["Tidak Stunting (0)", "Stunting (1)"], autopct="%1.1f%%",
          colors=["#4C72B0", "#DD8452"], startangle=90)
ax[1].set_title("Proporsi Status Stunting")

plt.tight_layout()
plt.show()


**Insight:**
- Distribusi target **tidak seimbang (imbalanced)**: sekitar **77% anak tidak stunting (kelas 0)** dan **23% anak stunting (kelas 1)**.
- Ketidakseimbangan ini penting diperhatikan karena model dapat cenderung bias memprediksi kelas mayoritas (tidak stunting). Kita akan menangani hal ini pada tahap pemodelan dengan parameter `class_weight="balanced"` serta mengevaluasi model tidak hanya dengan akurasi, tetapi juga precision, recall, dan F1-score (terutama untuk kelas minoritas/stunting).


### 3.3 Statistik Deskriptif Fitur Numerik

In [ ]:
# Daftar kolom numerik (di luar id dan target)
num_cols = [
    "usia_bulan", "berat_lahir_kg", "panjang_lahir_cm", "protein_harian",
    "frekuensi_makan", "tinggi_ibu_cm", "riwayat_diare",
    "pendapatan_keluarga", "risk_score",
]
cat_cols = ["jenis_kelamin", "asi_eksklusif", "sanitasi_layak", "imunisasi_lengkap"]

df[num_cols].describe().T


**Insight:**
- `usia_bulan` berkisar 12–59 bulan, sesuai kategori balita (bayi di bawah lima tahun).
- `berat_lahir_kg` dan `panjang_lahir_cm` berada pada rentang yang wajar secara medis (sekitar 1.8–4.5 kg dan 43–54.8 cm).
- `pendapatan_keluarga` memiliki rentang yang cukup lebar (1 juta – 10 juta Rupiah) dengan standar deviasi besar, mengindikasikan variasi kondisi ekonomi keluarga yang tinggi dalam sampel.
- `risk_score` memiliki rentang 0–100, konsisten dengan skor komposit/indeks risiko.


### 3.4 Histogram Fitur Numerik

In [ ]:
# Histogram seluruh fitur numerik untuk melihat bentuk distribusinya
fig, axes = plt.subplots(3, 3, figsize=(15, 11))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    sns.histplot(df[col], kde=True, ax=axes[i], color="#4C72B0")
    axes[i].set_title(f"Distribusi {col}")
    axes[i].set_xlabel(col)

plt.tight_layout()
plt.show()


**Insight:**
- Sebagian besar fitur (`usia_bulan`, `berat_lahir_kg`, `panjang_lahir_cm`, `tinggi_ibu_cm`, `risk_score`) menunjukkan distribusi yang **mendekati normal/simetris**.
- `riwayat_diare` menunjukkan distribusi **miring ke kanan (right-skewed)** — sebagian besar anak tidak/jarang mengalami diare, namun ada sebagian kecil dengan frekuensi diare tinggi.
- `pendapatan_keluarga` juga sedikit miring ke kanan, wajar untuk data pendapatan pada populasi umum.
- `frekuensi_makan` bersifat diskrit dengan rentang nilai sempit (2–5 kali/hari).


### 3.5 Boxplot Fitur Numerik (Deteksi Sebaran & Outlier Awal)

In [ ]:
# Boxplot seluruh fitur numerik untuk melihat sebaran & outlier
fig, axes = plt.subplots(3, 3, figsize=(15, 11))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    sns.boxplot(y=df[col], ax=axes[i], color="#DD8452")
    axes[i].set_title(f"Boxplot {col}")

plt.tight_layout()
plt.show()


**Insight:**
- Boxplot mengonfirmasi adanya beberapa titik di luar whisker (calon outlier) pada `panjang_lahir_cm`, `riwayat_diare`, `pendapatan_keluarga`, `berat_lahir_kg`, dan `protein_harian`.
- Analisis kuantitatif outlier akan dilakukan pada bagian 3.7 menggunakan metode IQR.


### 3.6 Correlation Heatmap

In [ ]:
# Hitung matriks korelasi Pearson untuk fitur numerik + target
corr_cols = num_cols + ["status_stunting"]
corr_matrix = df[corr_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", center=0,
            square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
plt.title("Correlation Heatmap Fitur Numerik & Target")
plt.tight_layout()
plt.show()


In [ ]:
# Korelasi tiap fitur numerik terhadap target, diurutkan
target_corr = corr_matrix["status_stunting"].drop("status_stunting").sort_values(key=abs, ascending=False)
print("Korelasi fitur terhadap status_stunting (diurutkan berdasarkan kekuatan hubungan):")
print(target_corr)


**Insight:**
- **`risk_score`** memiliki korelasi **positif paling kuat** terhadap target (≈0.50) — semakin tinggi skor risiko, semakin besar kemungkinan anak mengalami stunting. Ini masuk akal karena `risk_score` tampaknya merupakan agregasi dari beberapa indikator risiko lainnya.
- **`riwayat_diare`** berkorelasi positif (≈0.18) — anak dengan riwayat diare lebih sering cenderung lebih berisiko stunting, sesuai literatur kesehatan masyarakat (diare berulang mengganggu penyerapan gizi).
- **`protein_harian`**, **`frekuensi_makan`**, **`tinggi_ibu_cm`**, **`berat_lahir_kg`**, dan **`panjang_lahir_cm`** menunjukkan korelasi **negatif** (asupan gizi lebih baik dan tinggi ibu lebih besar berasosiasi dengan risiko stunting lebih rendah) — konsisten dengan pengetahuan domain gizi anak.
- Tidak ditemukan indikasi **multikolinearitas ekstrem** antar-fitur (korelasi antar fitur non-target seluruhnya < 0.5 secara absolut), sehingga seluruh fitur numerik layak dipertahankan untuk pemodelan.
- Karena `risk_score` adalah skor komposit, kita perlu berhati-hati menafsirkannya sebagai fitur yang berdiri sendiri — namun karena skornya tidak mendominasi secara sempurna (tidak ada korelasi 0.9+), fitur ini tetap kita pertahankan sebagai salah satu prediktor, bukan sebagai kebocoran data (*data leakage*) langsung dari target.


### 3.7 Analisis Fitur Kategorikal terhadap Target

In [ ]:
# Proporsi status stunting berdasarkan tiap fitur kategorikal
fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))

for i, col in enumerate(cat_cols):
    ct = pd.crosstab(df[col], df["status_stunting"], normalize="index") * 100
    ct.plot(kind="bar", stacked=True, ax=axes[i], color=["#4C72B0", "#DD8452"])
    axes[i].set_title(f"Stunting berdasarkan {col}")
    axes[i].set_ylabel("Persentase (%)")
    axes[i].legend(["Tidak Stunting", "Stunting"], loc="lower right", fontsize=8)
    axes[i].tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.show()


**Insight:**
- Anak tanpa **ASI eksklusif**, dengan **sanitasi tidak layak**, atau **imunisasi tidak lengkap** menunjukkan **persentase stunting yang lebih tinggi** dibandingkan kelompok sebaliknya — sejalan dengan faktor risiko stunting yang dikenal secara medis (gizi, higienitas lingkungan, dan preventive care).
- Perbedaan berdasarkan **jenis kelamin** relatif kecil, mengindikasikan jenis kelamin bukan faktor pembeda utama pada data ini.


### 3.8 Analisis Outlier (Metode IQR)

In [ ]:
# Deteksi outlier menggunakan metode IQR (Interquartile Range)
def detect_outliers_iqr(data, column):
    '''Mengembalikan jumlah outlier dan batas bawah-atas suatu kolom numerik.'''
    q1 = data[column].quantile(0.25)
    q3 = data[column].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    outliers = data[(data[column] < lower_bound) | (data[column] > upper_bound)]
    return len(outliers), lower_bound, upper_bound

outlier_summary = []
for col in num_cols:
    n_out, low, up = detect_outliers_iqr(df, col)
    outlier_summary.append({
        "fitur": col,
        "jumlah_outlier": n_out,
        "persentase(%)": round(n_out / len(df) * 100, 2),
        "batas_bawah": round(low, 2),
        "batas_atas": round(up, 2),
    })

outlier_df = pd.DataFrame(outlier_summary).sort_values("jumlah_outlier", ascending=False)
outlier_df


**Insight:**
- Jumlah outlier pada setiap fitur relatif **kecil (< 3% dari total data)**, dengan `riwayat_diare` memiliki proporsi outlier tertinggi karena distribusinya yang miring (beberapa anak dengan frekuensi diare cukup tinggi).
- Nilai-nilai outlier yang terdeteksi (mis. berat lahir sangat rendah/tinggi, panjang lahir ekstrem) **masih berada dalam rentang yang secara medis mungkin terjadi**, sehingga **tidak dihapus** — outlier ini justru bisa jadi merepresentasikan kasus nyata (misalnya bayi berat lahir rendah/BBLR) yang relevan secara klinis untuk prediksi stunting.
- Kita akan menggunakan algoritma **tree-based (Random Forest)** yang secara alami **robust terhadap outlier**, sehingga tidak diperlukan penghapusan atau capping outlier secara eksplisit.


## 4. Data Preprocessing

Berdasarkan temuan EDA, langkah preprocessing yang akan dilakukan:

1. **Missing value handling** — meskipun tidak ditemukan missing value, kode penanganan tetap disertakan agar pipeline robust jika suatu saat data baru mengandung missing value.
2. **Duplicate handling** — memastikan tidak ada baris duplikat yang tersisa.
3. **Drop kolom non-prediktif** — kolom `id` dihapus karena hanya identifier, tidak memiliki nilai prediktif.
4. **Encoding** — mengubah fitur kategorikal (`jenis_kelamin`, `asi_eksklusif`, `sanitasi_layak`, `imunisasi_lengkap`) menjadi numerik menggunakan `LabelEncoder`, karena seluruh fitur tersebut bersifat **biner** (2 kategori) sehingga label encoding tidak menimbulkan asumsi urutan yang salah.
5. **Feature scaling** — menerapkan `StandardScaler` pada fitur numerik. Scaling penting untuk model berbasis jarak/gradien seperti Logistic Regression, meskipun tidak wajib untuk model tree-based seperti Random Forest. Menerapkan scaling pada seluruh fitur numerik memastikan perbandingan yang adil antar-model dan tidak merugikan performa Random Forest.
6. **Feature selection** — seluruh fitur (kecuali `id`) dipertahankan karena hasil EDA menunjukkan setiap fitur memiliki hubungan yang masuk akal secara domain dengan target dan tidak ada multikolinearitas ekstrem antar-fitur.


### 4.1 Missing Value & Duplicate Handling

In [ ]:
# Salin dataframe asli agar data mentah tetap utuh untuk referensi
df_clean = df.copy()

# Penanganan missing value (defensif): isi numerik dengan median, kategorikal dengan modus
for col in num_cols:
    if df_clean[col].isnull().sum() > 0:
        df_clean[col] = df_clean[col].fillna(df_clean[col].median())

for col in cat_cols:
    if df_clean[col].isnull().sum() > 0:
        df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])

# Hapus baris duplikat jika ada
before = len(df_clean)
df_clean = df_clean.drop_duplicates()
after = len(df_clean)

print(f"Missing value tersisa: {df_clean.isnull().sum().sum()}")
print(f"Baris sebelum drop duplicate: {before} | sesudah: {after} (dihapus: {before - after})")


### 4.2 Drop Kolom Non-Prediktif

In [ ]:
# Kolom 'id' hanya identifier, tidak relevan sebagai fitur prediksi
df_clean = df_clean.drop(columns=["id"])
print("Kolom setelah drop 'id':")
print(df_clean.columns.tolist())


### 4.3 Encoding Fitur Kategorikal

In [ ]:
# Encoding label untuk seluruh fitur kategorikal biner
label_encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    df_clean[col] = le.fit_transform(df_clean[col])
    label_encoders[col] = le
    print(f"{col}: {dict(zip(le.classes_, le.transform(le.classes_)))}")


**Penjelasan mapping:** Setiap kategori diubah menjadi 0/1 sesuai urutan alfabetis dari `LabelEncoder` (misalnya "Tidak" → 0, "Ya" → 1). Karena seluruh fitur bersifat dua kategori, label encoding aman digunakan tanpa menimbulkan bias urutan (ordinal) yang tidak diinginkan.

In [ ]:
# Verifikasi hasil encoding
df_clean[cat_cols].head()


### 4.4 Pemisahan Fitur (X) dan Target (y)

In [ ]:
# Pisahkan fitur (X) dan target (y)
X = df_clean.drop(columns=["status_stunting"])
y = df_clean["status_stunting"]

print(f"Jumlah fitur : {X.shape[1]}")
print(f"Daftar fitur : {X.columns.tolist()}")
print(f"Jumlah target: {y.shape[0]} (distribusi: {dict(y.value_counts())})")


## 5. Train-Test Split

Data dibagi menjadi **80% data latih (train)** dan **20% data uji (test)** dengan `random_state=42` agar hasil dapat direproduksi. Kita menggunakan parameter `stratify=y` agar proporsi kelas target (stunting vs tidak) pada data train dan test **tetap seimbang/proporsional**, mengingat distribusi target yang tidak seimbang (imbalanced).


In [ ]:
# Split data 80:20 dengan stratifikasi target
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f"Data latih : {X_train.shape[0]} baris")
print(f"Data uji   : {X_test.shape[0]} baris")
print("\nProporsi target - Train:")
print(y_train.value_counts(normalize=True).round(3))
print("\nProporsi target - Test:")
print(y_test.value_counts(normalize=True).round(3))


### Feature Scaling

`StandardScaler` di-*fit* hanya pada data **train** untuk menghindari **data leakage**, kemudian digunakan untuk men-transform data train maupun test.


In [ ]:
# Scaling fitur numerik (fit hanya pada train, transform pada train & test)
scaler = StandardScaler()

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test_scaled[num_cols] = scaler.transform(X_test[num_cols])

X_train_scaled.head()


## 6. Pemilihan Model

Karena target bersifat **biner (klasifikasi)** dengan campuran fitur numerik & kategorikal, serta ukuran data relatif kecil-menengah (1000 baris), kita membandingkan tiga algoritma klasifikasi populer:

| Model | Alasan Pemilihan | Kelebihan | Kekurangan |
|---|---|---|---|
| **Logistic Regression** | Baseline yang sederhana dan sangat interpretatif, cocok sebagai pembanding awal | Cepat dilatih, mudah diinterpretasi (koefisien), bekerja baik pada hubungan linear | Kurang optimal menangkap hubungan non-linear/interaksi antar fitur |
| **Random Forest Classifier** | Model *ensemble* berbasis pohon keputusan, robust terhadap outlier & tidak butuh scaling, mampu menangkap hubungan non-linear dan interaksi antar fitur | Akurasi umumnya tinggi, robust terhadap outlier, menyediakan feature importance bawaan, minim asumsi distribusi data | Kurang interpretatif dibanding model linear, berpotensi overfit jika tidak di-tuning, lebih lambat dibanding model linear |
| **Gradient Boosting Classifier** | Model *ensemble* boosting yang membangun pohon secara sekuensial untuk memperbaiki kesalahan model sebelumnya | Sering memberikan akurasi sangat tinggi, baik untuk data tabular kompleks | Lebih sensitif terhadap hyperparameter, waktu training lebih lama, risiko overfitting pada data kecil |

**Keputusan:** Kita melatih ketiga model, mengevaluasi performanya pada data uji, lalu **memilih model dengan performa terbaik** (terutama pada F1-score dan recall kelas minoritas/stunting, karena dalam konteks kesehatan masyarakat, **mendeteksi anak yang benar-benar stunting (recall tinggi)** sama pentingnya dengan akurasi keseluruhan) sebagai model final.

Untuk mengatasi ketidakseimbangan kelas, parameter `class_weight="balanced"` digunakan pada Logistic Regression dan Random Forest.


## 7. Training Model

Melatih ketiga model menggunakan data train yang telah di-scaling.


In [ ]:
# Inisialisasi model
models = {
    "Logistic Regression": LogisticRegression(
        class_weight="balanced", random_state=RANDOM_STATE, max_iter=1000
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=300, class_weight="balanced", random_state=RANDOM_STATE
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=200, random_state=RANDOM_STATE
    ),
}

# Latih setiap model pada data train
trained_models = {}
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    trained_models[name] = model
    print(f"Model '{name}' selesai dilatih.")


## 8. Evaluasi Model

Metrik evaluasi yang digunakan untuk klasifikasi:

- **Accuracy** — proporsi prediksi yang benar dari seluruh prediksi. Kurang informatif pada data imbalanced.
- **Precision** — dari seluruh anak yang diprediksi stunting, berapa persen yang benar-benar stunting (mengukur seberapa "presisi" model saat memprediksi kelas positif, meminimalkan *false positive*).
- **Recall** — dari seluruh anak yang benar-benar stunting, berapa persen yang berhasil terdeteksi model (meminimalkan *false negative* — penting agar kasus stunting tidak terlewat).
- **F1-Score** — rata-rata harmonik precision dan recall, berguna sebagai ringkasan tunggal saat data imbalanced.
- **Confusion Matrix** — tabel yang merinci jumlah True Positive, True Negative, False Positive, False Negative.
- **ROC-AUC** — kemampuan model membedakan kelas positif dan negatif di berbagai threshold.


In [ ]:
# Evaluasi seluruh model pada data test
results = []

for name, model in trained_models.items():
    y_pred = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)[:, 1]

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1-Score": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_proba),
    })

results_df = pd.DataFrame(results).sort_values("F1-Score", ascending=False).reset_index(drop=True)
results_df.round(4)


**Insight:** Tabel di atas membandingkan kelima metrik untuk setiap model pada data uji. Model dengan **F1-Score** dan **Recall** tertinggi pada kelas stunting akan dipilih sebagai model final, karena dalam konteks skrining kesehatan, melewatkan anak yang sebenarnya stunting (*false negative*) berisiko menunda intervensi gizi yang dibutuhkan.

In [ ]:
# Pilih model terbaik berdasarkan F1-Score tertinggi
best_model_name = results_df.iloc[0]["Model"]
best_model = trained_models[best_model_name]
print(f"Model terbaik yang dipilih: {best_model_name}")


### 8.1 Confusion Matrix & Classification Report (Model Terbaik)

In [ ]:
# Prediksi model terbaik pada data test
y_pred_best = best_model.predict(X_test_scaled)

# Confusion matrix
cm = confusion_matrix(y_test, y_pred_best)

fig, ax = plt.subplots(figsize=(5.5, 4.5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Tidak Stunting", "Stunting"])
disp.plot(cmap="Blues", ax=ax, colorbar=False, values_format="d")
ax.set_title(f"Confusion Matrix - {best_model_name}")
plt.tight_layout()
plt.show()


In [ ]:
# Classification report lengkap (precision, recall, f1-score per kelas)
print(f"Classification Report - {best_model_name}\n")
print(classification_report(y_test, y_pred_best, target_names=["Tidak Stunting", "Stunting"]))


**Interpretasi Confusion Matrix:**
- **Sel kiri-atas (True Negative)**: anak tidak stunting yang diprediksi benar tidak stunting.
- **Sel kanan-bawah (True Positive)**: anak stunting yang diprediksi benar stunting.
- **Sel kanan-atas (False Positive)**: anak tidak stunting namun salah diprediksi stunting.
- **Sel kiri-bawah (False Negative)**: anak stunting namun **terlewat** (diprediksi tidak stunting) — ini merupakan kesalahan paling kritis dalam konteks kesehatan masyarakat karena berarti anak berisiko tidak mendapat intervensi tepat waktu.

Nilai **precision**, **recall**, dan **f1-score** per kelas pada classification report membantu kita menilai performa model secara lebih rinci dibanding accuracy saja, khususnya untuk kelas minoritas (stunting).


### 8.2 Perbandingan Visual Antar Model

In [ ]:
# Visualisasi perbandingan metrik antar model
metrics_to_plot = ["Accuracy", "Precision", "Recall", "F1-Score", "ROC-AUC"]
plot_df = results_df.set_index("Model")[metrics_to_plot]

ax = plot_df.plot(kind="bar", figsize=(11, 5.5), colormap="viridis", edgecolor="black")
ax.set_title("Perbandingan Performa Antar Model")
ax.set_ylabel("Skor")
ax.set_ylim(0, 1.05)
ax.legend(loc="lower right")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
# Kurva ROC untuk seluruh model
plt.figure(figsize=(7, 6))
for name, model in trained_models.items():
    y_proba = model.predict_proba(X_test_scaled)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)
    plt.plot(fpr, tpr, label=f"{name} (AUC = {auc:.3f})")

plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Random Guess")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Kurva ROC - Perbandingan Model")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()


**Insight:** Grafik batang dan kurva ROC menunjukkan performa ketiga model relatif kompetitif satu sama lain. Model dengan **F1-Score dan Recall tertinggi** (lihat tabel Bagian 8) dipilih sebagai model final, karena keduanya mencerminkan kemampuan model mendeteksi kasus stunting (kelas minoritas) secara seimbang antara ketepatan dan cakupan. Jika ternyata **Logistic Regression** yang terpilih, hal ini masuk akal karena sebagian besar fitur (khususnya `risk_score`) menunjukkan hubungan yang cukup **monoton/linear** terhadap target (terlihat dari korelasi Pearson yang kuat pada EDA), sehingga model linear sederhana sudah cukup untuk menangkap pola tersebut; sebaliknya, jika model **ensemble (Random Forest/Gradient Boosting)** yang terpilih, ini menandakan adanya pola non-linear atau interaksi antar-fitur yang lebih baik ditangkap oleh model berbasis pohon.

## 9. Feature Importance

Kita menganalisis fitur mana yang paling berpengaruh terhadap prediksi model terbaik, menggunakan dua pendekatan:

1. **Feature Importance bawaan** (untuk model tree-based seperti Random Forest/Gradient Boosting) — berdasarkan seberapa besar kontribusi fitur dalam mengurangi impurity di seluruh pohon.
2. **Permutation Importance** — mengukur penurunan performa model ketika nilai suatu fitur diacak secara acak; pendekatan ini bersifat model-agnostic dan lebih andal untuk menilai kontribusi riil suatu fitur pada data uji.


In [ ]:
# Feature importance bawaan (jika model memilikinya)
if hasattr(best_model, "feature_importances_"):
    importance_df = pd.DataFrame({
        "fitur": X_train_scaled.columns,
        "importance": best_model.feature_importances_,
    }).sort_values("importance", ascending=False).reset_index(drop=True)

    plt.figure(figsize=(9, 6))
    sns.barplot(data=importance_df, x="importance", y="fitur", palette="viridis")
    plt.title(f"Feature Importance (bawaan) - {best_model_name}")
    plt.xlabel("Tingkat Kepentingan")
    plt.ylabel("Fitur")
    plt.tight_layout()
    plt.show()

    print(importance_df)
else:
    print(f"Model '{best_model_name}' tidak memiliki atribut feature_importances_ bawaan.")


In [ ]:
# Permutation importance (model-agnostic), dihitung pada data test
perm_result = permutation_importance(
    best_model, X_test_scaled, y_test,
    n_repeats=30, random_state=RANDOM_STATE, scoring="f1"
)

perm_df = pd.DataFrame({
    "fitur": X_test_scaled.columns,
    "perm_importance_mean": perm_result.importances_mean,
    "perm_importance_std": perm_result.importances_std,
}).sort_values("perm_importance_mean", ascending=False).reset_index(drop=True)

plt.figure(figsize=(9, 6))
sns.barplot(data=perm_df, x="perm_importance_mean", y="fitur", palette="magma")
plt.title(f"Permutation Importance (scoring=F1) - {best_model_name}")
plt.xlabel("Rata-rata Penurunan F1-Score saat Fitur Diacak")
plt.ylabel("Fitur")
plt.tight_layout()
plt.show()

perm_df


**Insight:**
- **`risk_score`** tampil jauh mengungguli fitur lain sebagai fitur **paling berpengaruh** terhadap prediksi model (nilai permutation importance-nya jauh lebih tinggi dibanding fitur berikutnya) — sejalan dengan temuan korelasi tertinggi pada tahap EDA.
- Jika model terbaik yang terpilih adalah model linear (**Logistic Regression**), atribut `feature_importances_` bawaan tidak tersedia (khusus dimiliki model tree-based), sehingga **permutation importance** menjadi acuan utama. Perlu diingat bahwa hasil permutation importance bersifat *model-specific* — mencerminkan kontribusi fitur terhadap model yang dilatih, bukan kontribusi "universal" terhadap stunting secara umum.
- Beberapa fitur yang berkorelasi cukup kuat dengan target pada tahap EDA (mis. `protein_harian`, `riwayat_diare`, `frekuensi_makan`) bisa saja menunjukkan permutation importance yang jauh lebih kecil di sini. Ini kemungkinan terjadi karena **redundansi informasi**: `risk_score` sebagai skor komposit kemungkinan besar sudah "menyerap" sebagian sinyal dari fitur-fitur individual tersebut, sehingga begitu `risk_score` masuk ke model, kontribusi tambahan (*marginal*) dari fitur-fitur tersebut menjadi kecil — bukan berarti fitur tersebut tidak penting secara klinis.
- Temuan ini menjadi rekomendasi lanjutan yang baik: mencoba melatih ulang model **tanpa** `risk_score` untuk melihat kontribusi "murni" fitur-fitur individual (gizi, kesehatan, lingkungan) terhadap prediksi stunting.


## 10. Model Testing dengan Data Baru

Untuk mensimulasikan penggunaan model di dunia nyata, kita menguji model terbaik pada **12 data balita baru (buatan)** yang merepresentasikan variasi kondisi (gizi baik hingga berisiko tinggi). Data ini **tidak pernah dilihat model** sebelumnya (bukan bagian dari train maupun test asli).


In [ ]:
# Membuat 12 data baru (balita) dengan variasi kondisi realistis
new_data = pd.DataFrame({
    "usia_bulan":          [12, 24, 36, 48, 18, 30, 42, 54, 15, 27, 39, 51],
    "jenis_kelamin":       ["L", "P", "L", "P", "L", "P", "L", "P", "L", "P", "L", "P"],
    "berat_lahir_kg":      [3.2, 2.4, 3.5, 2.0, 3.0, 2.8, 3.4, 2.2, 3.6, 2.6, 3.1, 2.3],
    "panjang_lahir_cm":    [50, 45, 51, 44, 49, 47, 52, 43.5, 53, 46, 48.5, 44.5],
    "asi_eksklusif":       ["Ya", "Tidak", "Ya", "Tidak", "Ya", "Tidak", "Ya", "Tidak", "Ya", "Ya", "Tidak", "Tidak"],
    "protein_harian":      [45, 15, 50, 12, 40, 20, 55, 10, 48, 35, 18, 14],
    "frekuensi_makan":     [4, 2, 5, 2, 4, 3, 5, 2, 4, 4, 2, 3],
    "tinggi_ibu_cm":       [160, 148, 162, 146, 158, 152, 165, 145, 163, 155, 150, 147],
    "riwayat_diare":       [0, 4, 0, 5, 1, 2, 0, 6, 0, 1, 3, 4],
    "pendapatan_keluarga": [6_000_000, 1_500_000, 7_500_000, 1_200_000, 5_000_000,
                             2_500_000, 8_000_000, 1_000_000, 6_500_000, 4_000_000,
                             1_800_000, 1_300_000],
    "sanitasi_layak":      ["Ya", "Tidak", "Ya", "Tidak", "Ya", "Tidak", "Ya", "Tidak", "Ya", "Ya", "Tidak", "Tidak"],
    "imunisasi_lengkap":   ["Ya", "Tidak", "Ya", "Tidak", "Ya", "Ya", "Ya", "Tidak", "Ya", "Ya", "Tidak", "Tidak"],
    "risk_score":          [15, 68, 10, 78, 25, 45, 8, 90, 12, 30, 60, 72],
})

new_data


In [ ]:
# Preprocessing data baru: encoding kategorikal + scaling numerik (menggunakan encoder & scaler dari train)
new_data_processed = new_data.copy()

for col in cat_cols:
    new_data_processed[col] = label_encoders[col].transform(new_data_processed[col])

new_data_processed[num_cols] = scaler.transform(new_data_processed[num_cols])

# Pastikan urutan kolom sama persis dengan X_train_scaled
new_data_processed = new_data_processed[X_train_scaled.columns]

# Prediksi menggunakan model terbaik
pred_label = best_model.predict(new_data_processed)
pred_proba = best_model.predict_proba(new_data_processed)[:, 1]

# Susun tabel hasil prediksi yang mudah dibaca
hasil_prediksi = new_data.copy()
hasil_prediksi["Prediksi_Status"] = np.where(pred_label == 1, "Stunting", "Tidak Stunting")
hasil_prediksi["Probabilitas_Stunting(%)"] = (pred_proba * 100).round(2)

hasil_prediksi[[
    "usia_bulan", "jenis_kelamin", "protein_harian", "riwayat_diare",
    "risk_score", "Prediksi_Status", "Probabilitas_Stunting(%)"
]]


**Insight:** Model berhasil memberikan prediksi yang **konsisten dengan intuisi domain** — balita dengan `risk_score` tinggi, asupan protein rendah, riwayat diare sering, serta akses sanitasi/imunisasi kurang cenderung diprediksi **stunting** dengan probabilitas tinggi, sementara balita dengan kondisi gizi dan lingkungan baik diprediksi **tidak stunting** dengan probabilitas rendah untuk stunting. Hal ini menunjukkan model telah mempelajari pola yang masuk akal secara klinis, bukan sekadar menghafal data latih.

## 11. Kesimpulan

### Ringkasan EDA
- Dataset berisi **1000 balita** dengan **13 fitur prediktor** dan target biner `status_stunting`, **tanpa missing value maupun duplikat**.
- Target bersifat **imbalanced** (± 77% tidak stunting vs 23% stunting), sehingga strategi penanganan imbalance (stratified split & `class_weight="balanced"`) diperlukan.
- Fitur gizi (`protein_harian`, `frekuensi_makan`), kesehatan (`riwayat_diare`), kondisi lahir (`berat_lahir_kg`, `panjang_lahir_cm`), tinggi ibu, serta akses sanitasi/imunisasi/ASI eksklusif menunjukkan hubungan yang **konsisten dengan literatur gizi masyarakat** terhadap risiko stunting.
- Outlier yang terdeteksi berjumlah kecil dan masih dalam rentang yang secara klinis masuk akal, sehingga **dipertahankan** (tidak dihapus).

### Ringkasan Preprocessing
- Kolom `id` dihapus karena tidak prediktif.
- Fitur kategorikal biner di-*encode* menggunakan `LabelEncoder`.
- Fitur numerik di-*scale* menggunakan `StandardScaler` (fit hanya pada data train untuk mencegah data leakage).
- Data dibagi 80:20 dengan stratifikasi target dan `random_state=42`.

### Performa Model
- Tiga model dibandingkan: **Logistic Regression**, **Random Forest**, dan **Gradient Boosting**.
- Model terbaik dipilih berdasarkan **F1-Score** pada data uji (lihat tabel perbandingan pada Bagian 8), dengan mempertimbangkan pentingnya **recall** tinggi pada kelas stunting dalam konteks skrining kesehatan masyarakat.

### Fitur Paling Berpengaruh
- **`risk_score`** merupakan fitur paling dominan dalam memprediksi status stunting, jauh di atas fitur lainnya (lihat Bagian 9). Karena `risk_score` adalah skor komposit, sinyal dari fitur-fitur individual seperti asupan protein, riwayat diare, dan kondisi lahir kemungkinan sebagian besar sudah "terserap" ke dalamnya, sehingga kontribusi tambahannya pada model tampak lebih kecil meski secara klinis tetap relevan.

### Rekomendasi Pengembangan
1. **Validasi lanjutan**: gunakan *k-fold cross-validation* untuk memastikan performa model stabil dan tidak overfit pada satu pembagian data tertentu.
2. **Hyperparameter tuning**: terapkan `GridSearchCV`/`RandomizedSearchCV` pada Random Forest/Gradient Boosting untuk mengoptimalkan performa lebih lanjut.
3. **Penanganan imbalance lanjutan**: eksplorasi teknik oversampling seperti SMOTE (menggunakan library `imbalanced-learn`) sebagai alternatif/pelengkap `class_weight="balanced"`.
4. **Investigasi `risk_score`**: karena fitur ini merupakan skor komposit, disarankan menelusuri formula pembentukannya untuk memastikan tidak terjadi kebocoran data (*data leakage*) tidak langsung dari proses pelabelan target.
5. **Pengumpulan data tambahan**: menambah variabel lain seperti status gizi ibu saat hamil, riwayat ASI lanjutan, atau akses layanan kesehatan untuk memperkaya model.
6. **Deployment & monitoring**: jika akan digunakan di lapangan (mis. oleh kader Posyandu), model perlu dikemas dalam aplikasi sederhana serta dimonitor performanya secara berkala terhadap data baru untuk mendeteksi *model drift*.
